# Huấn luyện Mô hình Đánh giá IELTS 1-LoRA (Multi-Task Fine-Tuning với Unsloth)

Thực hiện **Giai đoạn 3: Huấn luyện 1-LoRA** cho dự án **Automated Essay Scoring (AES)**.
Chúng ta chuyển đổi từ kiến trúc 4 LoRA adapters riêng biệt sang **một LoRA adapter duy nhất (1-LoRA)** chấm điểm đồng thời cả 4 tiêu chí (TR, CC, LR, GRA) và trả về định dạng JSON thống nhất.

**Tối ưu hóa phần cứng VRAM:**
- Sử dụng thư viện **Unsloth** để tăng tốc độ huấn luyện gấp 2 lần và giảm 60% mức tiêu thụ VRAM.
- Lượng hóa 4-bit (`load_in_4bit=True`).
- Bộ tối ưu hóa `paged_adamw_8bit` giải phóng bộ nhớ khi đạt đỉnh.
- Gradient Checkpointing giúp tiết kiệm bộ nhớ KV Cache.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from dotenv import load_dotenv

# Nạp các biến môi trường từ tệp .env ở gốc dự án trước khi import Unsloth hoặc Transformers
load_dotenv(os.path.abspath("../.env"))

# --- BẢN VÁ LỖI UNPICKLING CHO PYTORCH ---
import torch
import numpy as np
try:
    from numpy.core.multiarray import _reconstruct
except ImportError:
    from numpy._core.multiarray import _reconstruct

try:
    from numpy import dtype, ndarray
except ImportError:
    dtype = np.dtype
    ndarray = np.ndarray

# Cho phép unpickle cả reconstruct, dtype, ndarray và một số kiểu dữ liệu số cơ bản
safe_globals = [_reconstruct, dtype, ndarray, np.float32, np.float64, np.int64]

# Tự động quét và đăng ký tất cả các lớp dtype trong module numpy.dtypes (như UInt32DType, v.v.)
try:
    import numpy.dtypes
    for name in dir(numpy.dtypes):
        attr = getattr(numpy.dtypes, name)
        if isinstance(attr, type):
            safe_globals.append(attr)
except (ImportError, AttributeError):
    pass
# Áp dụng đăng ký danh sách an toàn vào PyTorch
torch.serialization.add_safe_globals(safe_globals)
# ----------------------------------------

# IMPORT UNSLOTH TRƯỚC TRANSFORMERS để đảm bảo tối ưu hóa và tránh lỗi cảnh báo
from unsloth import FastLanguageModel
from unsloth import is_bfloat16_supported

# Import và áp dụng bản vá lỗi bảo mật check_torch_load_is_safe trước khi chạy huấn luyện
import transformers
import transformers.utils.import_utils
import transformers.utils
import transformers.trainer
import transformers.modeling_utils
import transformers.trainer_utils
for module in [
    transformers.utils.import_utils,
    transformers.utils,
    transformers.trainer,
    transformers.modeling_utils,
    transformers.trainer_utils
]:
    if hasattr(module, "check_torch_load_is_safe"):
        module.check_torch_load_is_safe = lambda: None

import sys
import json
import pandas as pd
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Thêm đường dẫn src/ để sử dụng rag_utils
sys.path.append(os.path.abspath("../src"))
from rag import rag_utils

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


## 1. Thiết lập Cấu hình & Tải Mô hình Llama-3.1-8B Lượng hóa 4-bit

In [2]:
max_seq_length = 1024 # Đủ chứa: New Essay (500 tokens) + 2 Reference Essays (1000 tokens) + Prompt & Output (500 tokens)
dtype = None           # Tự động phát hiện (Float16 hoặc Bfloat16)
load_in_4bit = True    # Bắt buộc bật để tiết kiệm VRAM trên card 8GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

from unsloth import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:05<00:00, 56.42it/s]
Unsloth: Will load unsloth/Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.


## 2. Thiết lập Cấu hình LoRA (PEFT)

Cấu hình các tham số rank `r=16` và `lora_alpha=32` để đảm bảo adapter học tốt các tác vụ đa mục tiêu mà không bị quá khớp.

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Tối ưu hóa bằng 0 trong Unsloth
    bias = "none",    # Tối ưu hóa none
    use_gradient_checkpointing = True, # Giảm thiểu tối đa VRAM tiêu thụ
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 3. Thiết kế Prompt Template & Tiền xử lý dữ liệu

### Tránh nghẽn RAG khi Train bằng phương pháp Pre-computation:
Thay vì truy vấn cơ sở dữ liệu Vector động trong quá trình train (làm giảm tốc độ train đi 10 lần), chúng ta thực hiện truy xuất RAG một lần cho tất cả mẫu Train/Val và lưu vào DataFrame tạm.

In [4]:
import gc
import torch

# Load Vector DB để truy xuất trước
VECTOR_DB_DIR = "../data/processed/chroma_db"
vectordb = rag_utils.load_vector_db(VECTOR_DB_DIR)

# Đọc tập dữ liệu sạch
df_train = pd.read_csv("../data/processed/train.csv")
df_val = pd.read_csv("../data/processed/val.csv")

# TỐI ƯU HÓA LOCAL: Lấy mẫu trước để chỉ chạy RAG trên 1,500 bài thay vì toàn bộ 7,460 bài!
df_train = df_train.sample(n=1500, random_state=42).reset_index(drop=True)
df_val = df_val.sample(n=200, random_state=42).reset_index(drop=True)

def add_rag_context_column(df):
    contexts = []
    for idx, row in df.iterrows():
        # Gọi trực tiếp k=2 và truyền tham số exclude_essay_text để chống leakage
        docs_to_use = rag_utils.retrieve_examples(vectordb, row['essay'], k=2, exclude_essay_text=row['essay'])
        context_str = rag_utils.format_rag_context(docs_to_use)
        contexts.append(context_str)
    df['rag_context'] = contexts
    return df
print("Đang truy xuất ngữ cảnh RAG cho tập Train...")
df_train = add_rag_context_column(df_train)
print("Đang truy xuất ngữ cảnh RAG cho tập Val...")
df_val = add_rag_context_column(df_val)

# Giải phóng bộ nhớ GPU sau khi hoàn thành truy xuất RAG
if 'vectordb' in locals() or 'vectordb' in globals():
    del vectordb
gc.collect()
torch.cuda.empty_cache()
print("✔ Đã giải phóng bộ nhớ GPU của Vector DB và hoàn thành RAG cho tập dữ liệu Local!")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3953.33it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
t:\5 - Summer 2026\AES_LLM\src\rag\rag_utils.py:28: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


Đang truy xuất ngữ cảnh RAG cho tập Train...
Đang truy xuất ngữ cảnh RAG cho tập Val...
✔ Đã giải phóng bộ nhớ GPU của Vector DB và hoàn thành RAG cho tập dữ liệu Local!


### Thiết kế cấu trúc Prompt & Sinh JSON đầu ra

#### Sửa lỗi rò rỉ dữ liệu (Data Leakage):
Prompt mẫu truyền vào lúc Train **không được chứa đáp án** (tr_band, tr_comment...). Nó phải dùng các placeholder trống giống hệt lúc Test (ví dụ: `<score>`, `<brief justification>`). Điều này buộc mô hình phải tự học cách chấm điểm từ Essay, chứ không phải đi copy đáp án nằm sẵn trong Prompt mẫu.

In [5]:
# Import template tập trung từ rag_utils thay vì khai báo cứng để tránh không đồng bộ
from rag.rag_utils import IELTS_EVAL_PROMPT_TEMPLATE
print("✔ Đã import thành công IELTS_EVAL_PROMPT_TEMPLATE tập trung!")

✔ Đã import thành công IELTS_EVAL_PROMPT_TEMPLATE tập trung!


In [6]:
EOS_TOKEN = tokenizer.eos_token

def clean_json_string(text):
    if pd.isna(text):
        return ""
    # Escape dấu ngoặc kép bằng 2 dấu gạch chéo ngược để ghi đè ký tự \ thực tế vào JSON
    return str(text).replace('"', '\\"').replace('\n', ' ').strip()

def formatting_prompts_func(examples):
    prompts = examples["prompt"]
    essays = examples["essay"]
    rag_contexts = examples["rag_context"]
    
    tr_bands = examples["TR_Band"]
    tr_comments = [clean_json_string(c) for c in examples["TR_Comment"]]
    
    cc_bands = examples["CC_Band"]
    cc_comments = [clean_json_string(c) for c in examples["CC_Comment"]]
    
    lr_bands = examples["LR_Band"]
    lr_mistakes = [str(m) if str(m).startswith('[') else '[]' for m in examples["LR_Mistakes"]]
    lr_corrections = [str(c) if str(c).startswith('[') else '[]' for c in examples["LR_Corrections"]]
    lr_comments = [clean_json_string(c) for c in examples["LR_Comment"]]
    
    gra_bands = examples["GRA_Band"]
    gra_mistakes = [str(m) if str(m).startswith('[') else '[]' for m in examples["GRA_Mistakes"]]
    gra_corrections = [str(c) if str(c).startswith('[') else '[]' for c in examples["GRA_Corrections"]]
    gra_comments = [clean_json_string(c) for c in examples["GRA_Comment"]]
    
    general_feedbacks = [clean_json_string(c) for c in examples["General_Feedback"]]
    
    texts = []
    for i in range(len(prompts)):
        question_str = f"Prompt: {prompts[i]}\nEssay: {essays[i]}"
        
        # Ghép nhãn đầu ra JSON
        response_json = (
            f'{{\n'
            f'  "Task_Response": {{\n'
            f'    "Band": {float(tr_bands[i])},\n'
            f'    "Comment": "{tr_comments[i]}"\n'
            f'  }},\n'
            f'  "Coherence_and_Cohesion": {{\n'
            f'    "Band": {float(cc_bands[i])},\n'
            f'    "Comment": "{cc_comments[i]}"\n'
            f'  }},\n'
            f'  "Lexical_Resource": {{\n'
            f'    "Band": {float(lr_bands[i])},\n'
            f'    "Mistakes": {lr_mistakes[i]},\n'
            f'    "Corrections": {lr_corrections[i]},\n'
            f'    "Comment": "{lr_comments[i]}"\n'
            f'  }},\n'
            f'  "Grammatical_Range_and_Accuracy": {{\n'
            f'    "Band": {float(gra_bands[i])},\n'
            f'    "Mistakes": {gra_mistakes[i]},\n'
            f'    "Corrections": {gra_corrections[i]},\n'
            f'    "Comment": "{gra_comments[i]}"\n'
            f'  }},\n'
            f'  "General_Feedback": "{general_feedbacks[i]}"\n'
            f'}}'
        )
        
        formatted_prompt = IELTS_EVAL_PROMPT_TEMPLATE.format(
            context=rag_contexts[i],
            question=question_str
        )
        
        # Sử dụng tokenizer.apply_chat_template để tự động gộp theo cấu trúc Llama 3.1
        messages = [
            {"role": "user", "content": formatted_prompt},
            {"role": "assistant", "content": response_json}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_pandas(df_train)
val_dataset = Dataset.from_pandas(df_val)

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
val_dataset = val_dataset.map(formatting_prompts_func, batched = True)
print("✔ Dữ liệu huấn luyện đã sẵn sàng!")

Map: 100%|██████████| 200/200 [00:00<00:00, 11018.36 examples/s]

✔ Dữ liệu huấn luyện đã sẵn sàng!


## 4. Thiết lập Huấn luyện viên (SFTTrainer)

In [7]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,      # Tối ưu hóa cực hạn VRAM
        per_device_eval_batch_size = 1,       # Tránh OOM khi chạy evaluation
        gradient_accumulation_steps = 8,     
        warmup_steps = 5,
        num_train_epochs = 1,                 # Chạy nhanh 1 epoch trên máy cá nhân
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),       
        logging_steps = 10,
        optim = "paged_adamw_8bit",           # Tiết kiệm bộ nhớ tối đa
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "../checkpoints",        # Thư mục lưu checkpoint sau mỗi epoch
        eval_strategy = "epoch", 
        save_strategy = "epoch",                               
        report_to = "none",
        # Tối ưu thêm VRAM
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
    ),
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 200/200 [00:00<00:00, 322.58 examples/s]


## 5. Bắt đầu Huấn luyện & Lưu Adapter

In [8]:
import sys
# VÁ LỖI: Định vị lại lớp SFTConfig trong sys.modules để tránh lỗi PicklingError khi lưu checkpoint
if 'trl.trainer.sft_config' in sys.modules:
    sys.modules['trl.trainer.sft_config'].SFTConfig = type(trainer.args)
    print("✔ Đã áp dụng bản vá lỗi Pickling cho SFTConfig!")

print("Đang huấn luyện mô hình...")
trainer_stats = trainer.train()

# Lưu LoRA Adapter cuối cùng
ADAPTER_DIR = "../adapters/llama_8b_1lora_aes"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✔ Đã lưu adapter tại: {ADAPTER_DIR}")

✔ Đã áp dụng bản vá lỗi Pickling cho SFTConfig!
Đang huấn luyện mô hình...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,500 | Num Epochs = 1 | Total steps = 188
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,1.739270,1.785630


t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\transformers\modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
t:\5 - Summer 2026\AES_LLM\.venv\lib\site-packages\transformers\modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Unsloth: Restored added_tokens_decoder metadata in ../checkpoints\checkpoint-188\tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ../adapters/llama_8b_1lora_aes\tokenizer_config.json.


✔ Đã lưu adapter tại: ../adapters/llama_8b_1lora_aes


## 6. Lưu Checkpoint Thủ công khi huấn luyện bị gián đoạn

Trong trường hợp quá trình huấn luyện bị ngắt quãng nửa chừng (do mất kết nối GPU, chủ động dừng bằng KeyboardInterrupt...), bạn có thể chạy ô lệnh dưới đây để lưu thủ công toàn bộ trạng thái hiện tại (bao gồm LoRA weights, optimizer state, scheduler state, RNG state...).    
Điều này đảm bảo bạn có thể khôi phục và chạy tiếp ở các phiên huấn luyện sau từ đúng bước này.

In [ ]:
import os
import torch
import random
import numpy as np

# 1. Xác định bước hiện tại và tạo thư mục
current_step = trainer.state.global_step
checkpoint_dir = f"../checkpoints/checkpoint-{current_step}"
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Đang tiến hành lưu thủ công checkpoint tại bước {current_step}...")

# 2. Lưu trọng số mô hình LoRA và Tokenizer
try:
    model.save_pretrained(checkpoint_dir)
    tokenizer.save_pretrained(checkpoint_dir)
    print("✔ Đã lưu model weights và tokenizer!")
except Exception as e:
    print(f"Lỗi lưu model: {e}")

# 3. Lưu trạng thái Optimizer
try:
    torch.save(trainer.optimizer.state_dict(), os.path.join(checkpoint_dir, "optimizer.pt"))
    print("✔ Đã lưu optimizer.pt!")
except Exception as e:
    print(f"Lỗi lưu optimizer: {e}")

# 4. Lưu trạng thái Scheduler
try:
    if trainer.lr_scheduler is not None:
        torch.save(trainer.lr_scheduler.state_dict(), os.path.join(checkpoint_dir, "scheduler.pt"))
        print("✔ Đã lưu scheduler.pt!")
except Exception as e:
    print(f"Lỗi lưu scheduler: {e}")

# 5. Lưu trạng thái Trainer State
try:
    trainer.state.save_to_json(os.path.join(checkpoint_dir, "trainer_state.json"))
    print("✔ Đã lưu trainer_state.json!")
except Exception as e:
    print(f"Lỗi lưu trainer_state: {e}")

# 6. Lưu cấu hình huấn luyện (training_args.bin)
try:
    torch.save(trainer.args, os.path.join(checkpoint_dir, "training_args.bin"))
    print("✔ Đã lưu training_args.bin!")
except Exception as e:
    print(f"Lỗi lưu training_args: {e}")

# 7. Lưu trạng thái RNG State (ngẫu nhiên)
try:
    checkpoint_rng_state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "cpu": torch.random.get_rng_state(),
    }
    if torch.cuda.is_available():
        checkpoint_rng_state["cuda"] = torch.cuda.get_rng_state_all()
    torch.save(checkpoint_rng_state, os.path.join(checkpoint_dir, "rng_state.pth"))
    print("✔ Đã lưu rng_state.pth!")
except Exception as e:
    print(f"Lỗi lưu rng_state: {e}")

print(f"\nThành công! Đã tạo xong checkpoint-{current_step} đầy đủ trạng thái.")

Đang tiến hành lưu thủ công checkpoint tại bước 188...


Unsloth: Restored added_tokens_decoder metadata in ../checkpoints/checkpoint-188\tokenizer_config.json.


✔ Đã lưu model weights và tokenizer!
✔ Đã lưu optimizer.pt!
✔ Đã lưu scheduler.pt!
✔ Đã lưu trainer_state.json!
✔ Đã lưu training_args.bin!
✔ Đã lưu rng_state.pth!

Thành công! Đã tạo xong checkpoint-188 đầy đủ trạng thái.
